# Palm Passport 02: when was each palm plot planted?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aldinorizaldy/geomata/blob/palm-passport-step2/notebooks/02_planting_year.ipynb)

Plots are dated from 40 years of Landsat. Clearing land for palm shows up as a sharp drop in
the NBR vegetation index, followed by regrowth as the canopy closes. LandTrendr finds that
drop on each 30 m pixel; the drop year (plus about one year) is the planting year. A
second drop means the plot was replanted.

**Outputs:** planting year, palm age, age class (0-3, 4-8, 9-25, over 25 = due for replanting),
and a replanting flag, plus a self-check against Hansen forest loss.

Run notebook 01 first if you haven't, and use the same `CENTER_LON` / `CENTER_LAT`.
**Runtime > Run all**. The map tiles are computed live, so give them a minute.

In [ ]:
# @title 0. Settings: fill in your Earth Engine project ID, then run all cells
PROJECT_ID = "your-ee-project-id"  # @param {type:"string"}
REPO_REF = "palm-passport-step2"  # @param {type:"string"}
BOX_KM = 20  # @param {type:"integer"}
# Leave both at 0 to auto-pick the 20 km box in Siak with the most smallholder palm.
CENTER_LON = 0.0  # @param {type:"number"}
CENTER_LAT = 0.0  # @param {type:"number"}

START_YEAR = 1985  # @param {type:"integer"}
END_YEAR = 2025  # @param {type:"integer"}
MIN_MAGNITUDE = 200  # @param {type:"integer"}
STATS_SCALE = 30  # @param {type:"integer"}
# Optional: Descals et al. 2024 planting-year map uploaded as an Earth Engine asset.
DESCALS_ASSET = ""  # @param {type:"string"}
DESCALS_BAND = "b1"  # @param {type:"string"}

In [ ]:
# @title 1. Install Palm Passport and sign in to Earth Engine
%pip install -q "palm-passport @ git+https://github.com/aldinorizaldy/geomata.git@$REPO_REF" geemap

import ee
import geemap.foliumap as geemap
import matplotlib.pyplot as plt
from palm_passport import aoi, ages, assets, auth, landsat, palm_mask, planting_year as py

auth.init(PROJECT_ID)

In [ ]:
# @title 2. Study area: Siak district and one 20 km test box
siak = aoi.district("Siak", "Riau")
if CENTER_LON and CENTER_LAT:
    study_box = aoi.box(CENTER_LON, CENTER_LAT, BOX_KM)
    print(f"Using your box centred on {CENTER_LON:.4f}, {CENTER_LAT:.4f}")
else:
    best = aoi.densest_smallholder_box(siak, BOX_KM).getInfo()
    props = best["properties"]
    study_box = ee.Geometry(best["geometry"])
    CENTER_LON, CENTER_LAT = props["center_lon"], props["center_lat"]
    print(f"Auto-picked box centred on {CENTER_LON:.4f}, {CENTER_LAT:.4f} "
          f"({props['smallholder_frac']:.0%} smallholder palm)")
    print("Tip: copy these numbers into CENTER_LON / CENTER_LAT so notebook 02 uses the same box.")
print(f"Siak district area: {siak.geometry().area(100).divide(1e6).getInfo():,.0f} km2")

In [ ]:
# @title 3. Data check: Landsat scenes per year over the box
counts = landsat.scene_counts(study_box, START_YEAR, END_YEAR).getInfo()["features"]
years = [f["properties"]["year"] for f in counts]
n = [f["properties"]["n_scenes"] for f in counts]
plt.figure(figsize=(10, 2.5))
plt.bar(years, n, color="#4575b4")
plt.title("Landsat scenes touching the box (before cloud masking)")
plt.ylabel("scenes"); plt.tight_layout(); plt.show()
print("Years with no scenes:", [y for y, c in zip(years, n) if c == 0] or "none")

In [ ]:
# @title 4. Yearly composites, LandTrendr and clearing dates (lazy: computed when displayed)
composites = landsat.annual_composites(study_box, START_YEAR, END_YEAR)
lt = py.run_landtrendr(composites, "NBR")
palm = palm_mask.palm_mask()
clearings = py.clearing_events(lt, min_magnitude=MIN_MAGNITUDE).updateMask(palm)
age = py.palm_age(clearings, as_of_year=END_YEAR + 1)
passport = age.addBands(clearings).addBands(palm_mask.oil_palm_classes()).clip(study_box)
print("Bands:", passport.bandNames().getInfo())

In [ ]:
# @title 5. Map: planting year, age class, replanting
def composite_for(year):
    return composites.filter(ee.Filter.eq("year", year)).first()

vis_rgb = {"bands": ["swir1", "nir", "red"], "min": 0, "max": 0.4}
m = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=13)
m.add_basemap("SATELLITE")
for y in [1990, 2005, END_YEAR]:
    m.addLayer(composite_for(y).clip(study_box), vis_rgb, f"Landsat {y} (SWIR-NIR-red)", False)
m.addLayer(passport.select("planting_year"),
           {"min": START_YEAR, "max": END_YEAR, "palette": ["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"]},
           "Planting year")
m.addLayer(passport.select("age_class"),
           {"min": 1, "max": 4, "palette": ages.AGE_CLASS_PALETTE}, "Age class", False)
m.addLayer(passport.select("replanted").selfMask(), {"palette": ["#ff00ff"]}, "Replanted", False)
m.addLayer(ee.Image().paint(ee.FeatureCollection([ee.Feature(study_box)]), 0, 3),
           {"palette": ["yellow"]}, "Test box")
m.add_colorbar({"min": START_YEAR, "max": END_YEAR,
                "palette": ["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"]},
               label="Planting year")
m.add_legend(title="Age class", legend_dict={
    label: color[1:] for (label, _, _), color in zip(ages.AGE_CLASSES, ages.AGE_CLASS_PALETTE)})
m

In [ ]:
# @title 6. Look inside one pixel: the NBR history LandTrendr saw
# Picks a random dated smallholder pixel; set your own lon/lat to inspect a specific plot.
PIXEL_LON = 0.0  # @param {type:"number"}
PIXEL_LAT = 0.0  # @param {type:"number"}

if PIXEL_LON and PIXEL_LAT:
    pt = ee.Geometry.Point([PIXEL_LON, PIXEL_LAT])
else:
    smallholder_dated = passport.select("planting_year").updateMask(passport.select("palm_class").eq(2))
    pt = smallholder_dated.sample(study_box, scale=30, numPixels=200, seed=7, geometries=True).first().geometry()

info = lt.select("LandTrendr").addBands(passport).reduceRegion(ee.Reducer.first(), pt, 30).getInfo()
yrs, raw, fit, _ = info["LandTrendr"]
plt.figure(figsize=(10, 3))
plt.plot(yrs, [-v / 1000 for v in raw], "o", color="#999", label="yearly NBR")
plt.plot(yrs, [-v / 1000 for v in fit], "-", color="#d95f02", lw=2, label="LandTrendr fit")
if info.get("last_clear_year"):
    plt.axvline(info["last_clear_year"], color="#1b9e77", ls="--", label=f"clearing {info['last_clear_year']}")
if info.get("replanted") and info.get("first_clear_year") != info.get("last_clear_year"):
    plt.axvline(info["first_clear_year"], color="#7570b3", ls=":", label=f"first clearing {info['first_clear_year']}")
plt.ylabel("NBR"); plt.legend(loc="lower left"); plt.title(f"Pixel at {pt.coordinates().getInfo()}")
plt.tight_layout(); plt.show()
print({k: info.get(k) for k in ["planting_year", "age", "age_class", "n_clearings", "palm_class"]})

In [ ]:
# @title 7. Summary: hectares by planting year and age class
area = ee.Image.pixelArea().divide(1e4)

def hist(band):
    d = area.addBands(passport.select(band)).reduceRegion(
        ee.Reducer.sum().group(1, band), study_box, STATS_SCALE, maxPixels=1e10, tileScale=8)
    return {int(g[band]): g["sum"] for g in d.getInfo()["groups"]}

by_year = hist("planting_year")
plt.figure(figsize=(10, 3))
plt.bar(list(by_year), list(by_year.values()), color="#21918c")
plt.title("Palm hectares by planting year (test box)"); plt.ylabel("ha")
plt.tight_layout(); plt.show()

by_class = hist("age_class")
for i, (label, _, _) in enumerate(ages.AGE_CLASSES, start=1):
    print(f"{label:30s} {by_class.get(i, 0):10,.0f} ha")
dated = sum(by_class.values())
palm_ha = area.updateMask(palm).reduceRegion(ee.Reducer.sum(), study_box, STATS_SCALE,
                                             maxPixels=1e10, tileScale=8).getInfo()["area"]
print(f"\nDated {dated:,.0f} of {palm_ha:,.0f} ha of palm ({dated / palm_ha:.0%}). "
      "Undated palm was probably cleared before the archive starts, or the drop was too small.")

In [ ]:
# @title 8. Self-check against Hansen forest loss (2001 onwards)
check = py.hansen_agreement(clearings, assets.hansen(), study_box, tolerance=1).getInfo()
print(f"Pixels where both maps see a clearing: {check['agree_count']:,}")
if check["agree_count"]:
    print(f"Share where clearing year matches Hansen loss year within 1 year: {check['agree_mean']:.0%}")

In [ ]:
# @title 9. (Optional) Benchmark against Descals et al. 2024 planting-year map
# Target from the spec (to confirm): within 2 years for 80% of points.
if DESCALS_ASSET:
    ref = ee.Image(DESCALS_ASSET).select(DESCALS_BAND).rename("ref_year")
    pts = passport.select("planting_year").addBands(ref).sample(
        study_box, scale=30, numPixels=5000, seed=1, dropNulls=True)
    diff = pts.map(lambda f: f.set("d", ee.Number(f.get("planting_year")).subtract(f.get("ref_year")).abs()))
    n = diff.size().getInfo()
    within2 = diff.filter(ee.Filter.lte("d", 2)).size().getInfo()
    print(f"{within2 / n:.0%} of {n} points within 2 years of Descals et al. 2024")
else:
    print("Skipped: set DESCALS_ASSET after uploading the Descals 2024 map to Earth Engine.")

In [ ]:
# @title 10. (Optional) Save the Palm Passport layers for the box to Google Drive
EXPORT = False  # @param {type:"boolean"}
if EXPORT:
    out = passport.select(["planting_year", "age", "age_class", "n_clearings", "first_clear_year",
                           "last_clear_year", "palm_class"]).toInt16()
    task = ee.batch.Export.image.toDrive(
        image=out, description="palm_passport_siak_planting_year", folder="palm_passport",
        region=study_box, scale=30, crs=aoi.UTM_47N, maxPixels=1e10)
    task.start()
    print("Export started. Watch it at https://code.earthengine.google.com/tasks")

## Known limits of this first version

- Landsat coverage over Sumatra before about 1990 is thin, and Riau is cloudy, so some early years are empty. Palm cleared before the archive starts has no date.
- Planting year = clearing year + 1. Real lags vary from 0 to 2 years.
- `MIN_MAGNITUDE` (200) and LandTrendr settings are starting values. Tune them with the Descals benchmark.
- Palm mask is the 2019 map: plots planted after 2019 are missing until we add a Sentinel-1/2 mask.